# Corponi Replication: Phase 2 (Supervised Fine-tuning on Hosseini/Indian Dataset)

## Overview
This notebook performs supervised fine-tuning on the Hosseini dataset using pre-trained weights from WESAD SSL Phase 1.

**IMPORTANT**: Cell 1 will restart the kernel to apply library updates. After restart, proceed with Cell 2.

In [ ]:
# 1. Environment Setup & Kernel Restart
import os, sys, shutil

WORK_DIR = '/kaggle/working/e4selflearning'
if os.path.exists(WORK_DIR): shutil.rmtree(WORK_DIR)
print("Cloning repository...")
!git clone https://github.com/april-tools/e4selflearning.git {WORK_DIR}

print("Installing dependencies (NumPy 1.26.4 compatibility)...")
!pip uninstall -y -q numpy scipy pandas
!pip install --no-cache-dir -q 'numpy==1.26.4' 'scipy==1.12.0' 'pandas<2.2' biopsykit flirt ruamel.yaml biosppy hrv-analysis neurokit2 distinctipy joypy peakutils lightning torchmetrics

print("\nRestarting kernel to apply changes...")
os._exit(0)

### 2. Raw Data Ingestion (Hosseini CSV to HDF5)
Converting the Indian dataset into Corponi-compatible HDF5 format.

In [ ]:
import os, sys, shutil, glob, pickle, numpy as np, pandas as pd
from tqdm import tqdm

if '/kaggle/working/e4selflearning/src' not in sys.path: sys.path.append('/kaggle/working/e4selflearning/src')
from timebase.utils import h5

def sub2id(sub):
    sub_clean = sub.replace('subject_', '').upper()
    if sub_clean == '1': return 27.0
    try: return float(ord(sub_clean[0]) - 64)
    except: return 0.0

RAW_DIR = '/kaggle/working/e4selflearning/data/preprocessed/unsegmented_hosseini'
if os.path.exists(RAW_DIR): shutil.rmtree(RAW_DIR)
os.makedirs(RAW_DIR, exist_ok=True)

subject_folders = sorted(glob.glob('/kaggle/input/**/subject_*', recursive=True))
if not subject_folders: raise FileNotFoundError("Hosseini subjects not found! Link the dataset as Input.")
dataset_root = os.path.dirname(subject_folders[0])

sessions_info = {}; task_map = {1: 1.0, 2: 1.0, 3: 1.0, 4: 0.0, 5: 0.0}
CHANNELS = ['ACC_x', 'ACC_y', 'ACC_z', 'BVP', 'EDA', 'TEMP', 'HR']
FREQ = {'BVP': 64, 'EDA': 4, 'TEMP': 4, 'ACC_x': 32, 'ACC_y': 32, 'ACC_z': 32, 'HR': 1}
global_data = {c: [] for c in CHANNELS}

print("Ingesting CSV files...")
for sub_folder in tqdm(sorted([d for d in os.listdir(dataset_root) if d.startswith('subject_')])):
    sub_path = os.path.join(dataset_root, sub_folder)
    sub_id = sub2id(sub_folder)
    for tid, status in task_map.items():
        task_dirs = glob.glob(os.path.join(sub_path, "**", f"T{tid}"), recursive=True)
        if not task_dirs: continue
        
        sess_id = f"{sub_folder}_T{tid}"; sess_dir = os.path.join(RAW_DIR, sess_id); os.makedirs(sess_dir, exist_ok=True)
        data_dict = {}; max_dur = 0
        for c in CHANNELS:
            f_name = c.split('_')[0] if 'ACC' in c else c
            files = glob.glob(os.path.join(task_dirs[0], f"*{f_name}*.csv"))
            if files:
                try:
                    df = pd.read_csv(files[0], header=None, skiprows=2, on_bad_lines='skip')
                    sig = df.iloc[:, {'x':0,'y':1,'z':2}[c.split('_')[1]] if 'ACC' in c else 0].values.astype(np.float32)
                    data_dict[c] = sig; global_data[c].append(sig)
                    max_dur = max(max_dur, len(sig) / FREQ[c])
                except: pass
        
        if max_dur < 60: 
            shutil.rmtree(sess_dir)
            continue
            
        duration = int(max_dur)
        for c in CHANNELS:
            target = duration * FREQ[c]
            if c not in data_dict: data_dict[c] = np.zeros(target, dtype=np.float32)
            else: data_dict[c] = np.pad(data_dict[c], (0, max(0, target - len(data_dict[c]))), mode='edge')[:target]
        
        # --- ADD DUMMY CHANNELS FOR CORPONI COMPATIBILITY ---
        data_dict['SLEEP'] = np.zeros(duration * 32, dtype=np.float32)
        data_dict['IBI'] = np.zeros(duration * 1, dtype=np.float32)
        
        labels = np.zeros(41, dtype=np.float32); labels[0] = sub_id; labels[3] = status
        data_dict['labels'] = labels
        h5.write(os.path.join(sess_dir, 'channels.h5'), data_dict, overwrite=True)
        sessions_info[sess_id] = {
            'channel_names': CHANNELS + ['IBI', 'SLEEP'], 
            'sampling_rates': {**FREQ, 'SLEEP': 32, 'IBI': 1}, 
            'unix_t0': {c: 0 for c in CHANNELS + ['IBI']}, 
            'Sub_ID': sub_id
        }

stats = {c: {'min': float(np.min(np.concatenate(v))), 'max': float(np.max(np.concatenate(v))), 'mean': float(np.mean(np.concatenate(v))), 'std': float(np.std(np.concatenate(v)))} if v else {'min':0,'max':1,'mean':0,'std':1} for c, v in global_data.items()}
with open(os.path.join(RAW_DIR, 'metadata.pkl'), 'wb') as f: pickle.dump({'sessions_info': sessions_info, 'stats': stats}, f)
print(f"Ingestion complete. Total sessions: {len(sessions_info)}")

### 3. Segmentation
Partitioning signals into 60-second segments (MASTER VERSION).

In [ ]:
SEG_DIR = '/kaggle/working/e4selflearning/data/preprocessed/sl60_ss30_hosseini'
os.makedirs(SEG_DIR, exist_ok=True)

print("Starting segmentation (60s windows, 30s step - MASTER VERSION)...")
!PYTHONPATH=/kaggle/working/e4selflearning/src python /kaggle/working/e4selflearning/segment.py \
    --data_dir {RAW_DIR} \
    --output_dir {SEG_DIR} \
    --segment_length 60 \
    --segmentation_mode 1 \
    --step_size 30 \
    --num_workers 2 \
    --overwrite

with open(os.path.join(SEG_DIR, 'stats.pkl'), 'wb') as f: pickle.dump(stats, f)
print("Segmentation finished.")

### 4. Custom Training Script
Generating the fixed trainer with Scale Mode 2 and 60s windows.

In [ ]:
script_content = r'''
import argparse, os, pickle, numpy as np, torch, re
from torch.utils.data import DataLoader
from tqdm import tqdm
from timebase import criterions
from timebase.models.models import get_models
from timebase.data.dataset import ClassificationDataset
from timebase.utils import utils
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix

class Indian_ClassificationDataset(ClassificationDataset):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.positive_classes = [1.0]

    def session_id(self, filename: str) -> int:
        match = re.search(r'subject_([A-Za-z0-9]+)', filename)
        sub = match.group(1).upper() if match else '0'
        return 27 if sub == '1' else (ord(sub[0]) - 64 if sub[0].isalpha() else 0)

def ensure_args_exists(path):
    yaml_path = os.path.join(path, "args.yaml")
    if not os.path.exists(yaml_path):
        print(f" [WARNING] args.yaml not found in {path}. Creating default config...")
        with open(yaml_path, "w") as f:
            f.write("task_mode: 1\nscaling_mode: 2\npretext_task: masked_prediction\nencoder_type: resnet\nrepresentation_module: resnet\nmask_ratio: 0.5\nsplit_mode: 0\ninclude_hr: False\nemb_dim: 64\nemb_num_filters: 4\nemb_type: 1\nmlp_dim: 64\nnum_blocks: 3\nnum_heads: 3\nnum_units: 64\na_dropout: 0.1\nm_dropout: 0.1\ndropout: 0.1\ndrop_path: 0.1\ndisable_bias: False\n")

def main(args):
    utils.set_random_seed(args.seed)
    args.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    os.makedirs(args.output_dir, exist_ok=True)
    
    args.verbose = 1; args.use_wandb = False; args.save_plots = False
    args.critic_score_lambda = 0.0; args.reuse_stats = False; args.clear_output_dir = False
    
    cli_dataset = args.dataset
    ensure_args_exists(args.path2pretraining_res)
    utils.load_args(args, dir=args.path2pretraining_res)
    args.dataset = cli_dataset; args.task_mode = 1; args.e4selflearning = True; args.scaling_mode = 2
    
    with open(os.path.join(args.dataset, "metadata.pkl"), "rb") as f: meta = pickle.load(f)
    
    CH_FREQ = {'BVP': 64, 'EDA': 4, 'TEMP': 4, 'ACC_x': 32, 'ACC_y': 32, 'ACC_z': 32}
    args.ds_info = {'channel_freq': CH_FREQ, 'segment_length': 60}
    args.input_shapes = {c: [60 * CH_FREQ[c]] for c in CH_FREQ.keys()}
    
    with open(os.path.join(args.dataset, "stats.pkl"), "rb") as f: stats = pickle.load(f)
    
    all_paths = np.array([os.path.join(args.dataset, os.path.basename(os.path.dirname(p)), os.path.basename(p)) for p in meta["sessions_paths"]])
    labels = {k: np.array(v) for k, v in meta["sessions_labels"].items()}
    
    # --- IMPROVED SUBJECT EXTRACTION ---
    unique_subs = sorted(list(set([re.search(r'subject_[A-Za-z0-9]+', p).group(0) for p in all_paths if "subject_" in p])))
    np.random.seed(args.seed); np.random.shuffle(unique_subs)
    split_idx = int(len(unique_subs) * 0.70)
    train_subs, test_subs = unique_subs[:split_idx], unique_subs[split_idx:]
    args.num_train_subjects = len(train_subs)
    
    m_train = np.array([any(s in p for s in train_subs) for p in all_paths])
    m_test = np.array([any(s in p for s in test_subs) for p in all_paths])
    
    ds_train = Indian_ClassificationDataset(args=args, filenames=all_paths[m_train], labels={k: v[m_train] for k, v in labels.items()}, rec_ids=all_paths[m_train], stats=stats, recording_id_str_to_num={p: i for i, p in enumerate(np.unique(all_paths[m_train]))})
    ds_test = Indian_ClassificationDataset(args=args, filenames=all_paths[m_test], labels={k: v[m_test] for k, v in labels.items()}, rec_ids=all_paths[m_test], stats=stats, recording_id_str_to_num={p: i for i, p in enumerate(np.unique(all_paths[m_test]))})
    
    loader_train = DataLoader(ds_train, batch_size=args.batch_size, shuffle=True)
    loader_test = DataLoader(ds_test, batch_size=args.batch_size, shuffle=False)
    
    classifier, _ = get_models(args, summary=None)
    target_file = None
    for r, d, f in os.walk(args.path2pretraining_res):
        for cand in ["model_state.pt", "model_best.pt"]: 
            if cand in f: target_file = os.path.join(r, cand); break
        if target_file: break
    
    if target_file:
        print(f" Loading weights from: {target_file}")
        ckpt = torch.load(target_file, map_location=args.device)
        sd = classifier.sslearner.state_dict()
        sd.update({k: v for k, v in ckpt["model"].items() if any(m in k for m in ["channel_embedding", "feature_encoder"])})
        classifier.sslearner.load_state_dict(sd)
    
    classifier.to(args.device)
    opt = torch.optim.AdamW(classifier.parameters(), lr=args.lr)
    crit = criterions.get_criterion(args)
    
    print(f" Training Subjects: {train_subs}\n Testing Subjects: {test_subs}")
    for epoch in range(1, args.epochs + 1):
        classifier.train(); losses = []
        for batch in tqdm(loader_train, desc=f"Epoch {epoch}", leave=False):
            inputs = {k: v.float().to(args.device) for k, v in batch["data"].items()}
            targets = batch["target"].to(args.device)
            out, _ = classifier(inputs)
            loss = crit.criterion_classifier(y_true=targets, y_pred=out)
            loss.backward(); opt.step(); opt.zero_grad()
            losses.append(loss.item())
        if epoch % 10 == 0 or epoch == 1: print(f" Epoch {epoch}: Loss = {np.mean(losses):.4f}")

    classifier.eval(); y_true, y_pred = [], []
    with torch.no_grad():
        for batch in tqdm(loader_test, desc="Evaluating"):
            inputs = {k: v.float().to(args.device) for k, v in batch["data"].items()}
            out, _ = classifier(inputs)
            y_true.extend(batch["target"].cpu().numpy())
            y_pred.extend((torch.sigmoid(out) >= 0.5).long().cpu().numpy())
    
    print("\n FINAL RESULTS HOSSEINI (MASTER VERSION):")
    print(f" Accuracy  : {accuracy_score(y_true, y_pred)*100:.2f}%")
    print(f" F1-Score  : {f1_score(y_true, y_pred, zero_division=0)*100:.2f}%")
    print(f" Precision : {precision_score(y_true, y_pred, zero_division=0)*100:.2f}%")
    print(f" Recall    : {recall_score(y_true, y_pred, zero_division=0)*100:.2f}%")
    torch.save(classifier.state_dict(), os.path.join(args.output_dir, "finetuned_hosseini.pt"))

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--dataset", type=str, required=True)
    parser.add_argument("--path2pretraining_res", type=str, required=True)
    parser.add_argument("--epochs", type=int, default=50)
    parser.add_argument("--batch_size", type=int, default=16)
    parser.add_argument("--lr", type=float, default=0.0001)
    parser.add_argument("--output_dir", type=str, required=True)
    parser.add_argument("--seed", type=int, default=42)
    main(parser.parse_args())
'''
with open("/kaggle/working/e4selflearning/train_ann_fixed.py", "w") as f: f.write(script_content)


### 5. Execution

In [ ]:
import os, glob, shutil
os.chdir('/kaggle/working/e4selflearning')

STAGE_DIR = '/kaggle/working/weights'
if os.path.exists(STAGE_DIR): shutil.rmtree(STAGE_DIR)
os.makedirs(os.path.join(STAGE_DIR, 'ckpt_sslearner'), exist_ok=True)

print("Searching for pre-trained weights and config in Input...")
ckpt_file = None; yaml_file = None
for r, d, f in os.walk('/kaggle/input'):
    for file in f:
        if file in ["model_state.pt", "model_best.pt"]: ckpt_file = os.path.join(r, file)
        if file == "args.yaml": yaml_file = os.path.join(r, file)

if ckpt_file:
    print(f"Found weights: {ckpt_file}")
    shutil.copy(ckpt_file, os.path.join(STAGE_DIR, 'ckpt_sslearner', 'model_state.pt'))
    if yaml_file:
        print(f"Found config: {yaml_file}")
        shutil.copy(yaml_file, os.path.join(STAGE_DIR, 'args.yaml'))

SEG_DIR = '/kaggle/working/e4selflearning/data/preprocessed/sl60_ss30_hosseini'

print(f"Starting Supervised Fine-Tuning (MASTER CONFIG)...")
!PYTHONPATH=/kaggle/working/e4selflearning/src python train_ann_fixed.py \
    --dataset {SEG_DIR} \
    --path2pretraining_res {STAGE_DIR} \
    --epochs 50 \
    --output_dir output_hosseini_final

### 6. Export Results

In [ ]:
import shutil
from IPython.display import FileLink
if os.path.exists("output_hosseini_final"):
    shutil.make_archive("hosseini_final_results", 'gztar', "output_hosseini_final")
    display(FileLink("hosseini_final_results.tar.gz"))